## Install Dependencies

This cell installs all required Python packages for the TechInterviewerAI workflow, including CrewAI, Playwright, crawl4ai, and ipywidgets for the UI.

In [ ]:
!pip install -qU crewai[tools]==0.95.0
!pip install -q playwright
!playwright install
!playwright install --with-deps
!pip install -q crawl4ai
!pip install -q langchain_openai
!pip install -q tavily-python
!pip install -q ipywidgets
!pip install -q nest_asyncio

## Import Libraries and Setup Environment

This cell imports necessary libraries, applies nest_asyncio for Colab compatibility, sets up API keys from Colab secrets, and creates an output directory for JSON files.

In [ ]:
import os
import json
import nest_asyncio
import ipywidgets as widgets
from IPython.display import display, HTML
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from pydantic import BaseModel, Field
from typing import List, Optional
from tavily import TavilyClient
from google.colab import userdata
from crawl4ai import AsyncWebCrawler
import asyncio
import smtplib
from email.mime.text import MIMEText

# Apply nest_asyncio for Colab compatibility
nest_asyncio.apply()

# Set up environment variables
os.environ["OPENAI_API_KEY"] = userdata.get('openai-colab')  # OpenAI API key
os.environ["TAVILY_API_KEY"] = userdata.get('tavily-colab')   # Tavily API key
os.environ["GMAIL_ADDRESS"] = userdata.get('gmail-address')  # Gmail address
os.environ["GMAIL_APP_PASSWORD"] = userdata.get('gmail-app-password')  # Gmail App Password

# Create output directory
output_dir = "./ai-agent-output"
os.makedirs(output_dir, exist_ok=True)

## Define Data Models

This cell defines Pydantic models for structuring the output of each agent, including job analysis, search queries, search results, scraped data, and interview questions.

In [ ]:
class JobAnalysis(BaseModel):
    job_technical_level: str = Field(..., title="Technical level of the job (entry, mid, senior)")
    key_skills: List[str] = Field(..., title="List of key technical skills required", min_items=1)
    include_ps: bool = Field(..., title="Whether to include problem-solving questions")
    domain_knowledge: List[str] = Field(..., title="List of domain knowledge areas", min_items=0)

class SearchQueries(BaseModel):
    search_queries: List[str] = Field(..., title="Search queries for interview questions", min_items=1)

class SingleSearchResult(BaseModel):
    title: str = Field(..., title="Title of the search result")
    url: str = Field(..., title="URL of the resource")
    content: str = Field(..., title="Snippet or summary of the resource content")
    score: float = Field(..., title="Relevance score of the result")
    search_query: str = Field(..., title="The query that generated this result")

class AllSearchResults(BaseModel):
    results: List[SingleSearchResult] = Field(..., title="List of search results")

class SkillDetail(BaseModel):
    name: str = Field(..., title="Name of the skill or technology")
    description: Optional[str] = Field(None, title="Brief description of the skill or technology")

class SingleScrapedPage(BaseModel):
    page_url: str = Field(..., title="The URL of the scraped webpage")
    skills: List[SkillDetail] = Field(default_factory=list, title="List of skills extracted")
    technologies: List[SkillDetail] = Field(default_factory=list, title="List of technologies extracted")
    question_examples: List[str] = Field(..., title="List of example interview questions", min_items=1)
    source_type: Optional[str] = Field(None, title="Type of source (e.g., job posting, interview guide)")

    @classmethod
    def check_minimum_data(cls, values):
        skills = values.get('skills', [])
        technologies = values.get('technologies', [])
        question_examples = values.get('question_examples', [])
        if not skills and not technologies and not question_examples:
            raise ValueError("At least one of skills, technologies, or question_examples must be non-empty")
        return values

class AllScrapedPages(BaseModel):
    pages: List[SingleScrapedPage] = Field(..., title="List of scraped webpages", min_items=1)

class InterviewQuestion(BaseModel):
    question: str = Field(..., title="Interview question in professional Egyptian Arabic")
    type: str = Field(..., title="Question category (technical, problem-solving, scenario-based)")
    difficulty: str = Field(..., title="Difficulty level (easy, medium, hard)")

class InterviewScript(BaseModel):
    questions: List[InterviewQuestion] = Field(..., title="List of interview questions", min_items=5, max_items=10)

## Define Tools

This cell defines the search and web scraping tools used by the agents, including a Tavily search tool and a crawl4ai-based web scraper.

In [ ]:
@tool
def search_engine_tool(query: str):
    """Search for resources related to technical skills and interview questions."""
    return search_client.search(query)

async def async_scrape_page(url: str) -> str:
    """Asynchronously scrape a webpage using crawl4ai."""
    try:
        print(f"Starting to scrape: {url}")
        async with AsyncWebCrawler() as crawler:
            result = await crawler.arun(url)
            if not result or not result.markdown:
                print("No content found when crawling the page")
                return "No content found on page"
            content = result.markdown
            print(f"Successfully crawled page with {len(content)} characters")
            return content
    except Exception as e:
        print(f"Error scraping page: {str(e)}")
        return f"Error scraping page: {str(e)}"

@tool
def web_scraping_tool_for_agent(page_url: str) -> str:
    """Synchronous wrapper for async scraping."""
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(async_scrape_page(page_url))

## Define Agents and Tasks

This cell defines the five agents (Input Processor, Search Query Generator, Research, Web Scraper, Question Generator) and their tasks, which process job details and generate interview questions.

In [ ]:
# Initialize LLMs
llm = LLM(model="gpt-4o-mini", temperature=0)
llm_advanced = LLM(model="gpt-4o", temperature=0)

# Initialize Tavily search client
search_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

# Agent A: Input Processor
input_processor_agent = Agent(
    role="Input Processor Agent",
    goal="Analyze job details to identify technical level, key skills, problem-solving relevance, and domain knowledge.",
    backstory="Starting point for TechInterviewerAI, supporting Arabic-speaking job seekers in the MENA tech market.",
    llm=llm,
    verbose=True,
)

input_processor_task = Task(
    description="Analyze job position, requirements, and company to produce a structured job analysis.",
    expected_output="JSON with job technical level, key skills, problem-solving relevance, and domain knowledge.",
    output_file=os.path.join(output_dir, "step_1_job_analysis.json"),
    output_json=JobAnalysis,
    agent=input_processor_agent
)

# Agent B: Search Query Generator
search_query_generator = Agent(
    role="Search Query Generator",
    goal="Generate tailored search queries for interview questions.",
    backstory="Specialized in creating effective search queries for job-specific interview questions.",
    llm=llm,
    verbose=True,
)

search_query_generator_task = Task(
    description="Generate up to 10 search queries based on job analysis.",
    expected_output="JSON with an array of search queries.",
    output_file=os.path.join(output_dir, "step_2_search_queries.json"),
    output_json=SearchQueries,
    agent=search_query_generator
)

# Agent C: Research Agent
research_agent = Agent(
    role="Research Agent",
    goal="Retrieve relevant resources for technical skills and interview questions.",
    backstory="Gathers high-quality resources for Arabic-speaking job seekers in the MENA tech market.",
    llm=llm,
    verbose=True,
    tools=[search_engine_tool]
)

research_task = Task(
    description="Search for resources using queries from step_2, prioritizing MENA relevance and score > {score_th}.",
    expected_output="JSON with search results including title, URL, content, score, and query.",
    output_json=AllSearchResults,
    output_file=os.path.join(output_dir, "step_3_research_results.json"),
    agent=research_agent
)

# Agent D: Web Scraper Agent
web_scraper_agent = Agent(
    role="Web Scraper Agent",
    goal="Scrape webpages to extract skills, technologies, and interview questions.",
    backstory="Collects and analyzes web data for TechInterviewerAI, supporting question generation.",
    llm=llm,
    verbose=True,
    tools=[web_scraping_tool_for_agent]
)

web_scraper_task = Task(
    description="Scrape URLs from step_3 to extract skills, technologies, and questions.",
    expected_output="JSON with scraped data including URL, skills, technologies, questions, and source type.",
    output_json=AllScrapedPages,
    output_file=os.path.join(output_dir, "step_4_scraped_data.json"),
    agent=web_scraper_agent
)

# Agent E: Question Generator
question_generator_agent = Agent(
    role="Enhanced Question Generator",
    goal="Craft interview questions rooted in job scenarios and MENA tech trends.",
    backstory="Powers TechInterviewerAI’s Q&A module, producing high-fidelity interview prompts.",
    llm=llm_advanced,
    verbose=True
)

question_generator_task = Task(
    description="Generate 5–10 interview questions in Egyptian Arabic based on step_4 data.",
    expected_output="JSON with 5–10 questions, each with question, type, and difficulty.",
    output_json=InterviewScript,
    output_file=os.path.join(output_dir, "step_5_interview_script.json"),
    agent=question_generator_agent
)

## Setup User Interface

This cell creates a UI using ipywidgets with input fields for job position, requirements, company name, score threshold, and email, plus a button to trigger the workflow.

In [ ]:
# Create input widgets
job_position_input = widgets.Text(
    value="",
    placeholder="e.g., R&D AI ML Developer Intern",
    description="Job Position:",
    layout={'width': '500px'}
)

requirements_input = widgets.Textarea(
    value="",
    placeholder="Paste job requirements here",
    description="Requirements:",
    layout={'width': '500px', 'height': '200px'}
)

company_name_input = widgets.Text(
    value="",
    placeholder="e.g., Siemens",
    description="Company Name:",
    layout={'width': '500px'}
)

score_th_input = widgets.FloatText(
    value=0.7,
    description="Score Threshold:",
    layout={'width': '500px'}
)

email_input = widgets.Text(
    value="",
    placeholder="e.g., user@example.com",
    description="User Email:",
    layout={'width': '500px'}
)

# Create submit button
submit_button = widgets.Button(
    description="Generate Interview Questions",
    button_style="primary",
    tooltip="Click to start the agent workflow",
    layout={'width': '200px'}
)

# Create output widget
output = widgets.Output()

# Display UI
display(HTML("<h2>TechInterviewerAI: Generate Interview Questions</h2>"))
display(job_position_input)
display(requirements_input)
display(company_name_input)
display(score_th_input)
display(email_input)
display(submit_button)
display(output)

Text(value='', description='Job Position:', layout=Layout(width='500px'), placeholder='e.g., R&D AI ML Develop…

Textarea(value='', description='Requirements:', layout=Layout(height='200px', width='500px'), placeholder='Pas…

Text(value='', description='Company Name:', layout=Layout(width='500px'), placeholder='e.g., Siemens')

FloatText(value=0.7, description='Score Threshold:', layout=Layout(width='500px'))

Text(value='', description='User Email:', layout=Layout(width='500px'), placeholder='e.g., user@example.com')

Button(button_style='primary', description='Generate Interview Questions', layout=Layout(width='200px'), style…

Output()

## Define Email Function

This cell defines a function to send an email with a meeting link to the user’s provided email address using Gmail’s SMTP server.

In [ ]:
def send_email(recipient, interview_script):
    """Send an email with a meeting link and interview questions to the recipient."""
    try:
        # Placeholder meeting link (replace with dynamic link if available)
        meeting_link = "https://zoom.us/j/123456789"  # Example Zoom link

        # Email content
        body = f"""
        Dear User,

        Your interview questions have been generated successfully. Below is a summary:

        {json.dumps(interview_script, indent=2, ensure_ascii=False)}

        A mock interview session has been scheduled. Please join using the following link:
        {meeting_link}

        Best regards,
        TechInterviewerAI Team
        """

        msg = MIMEText(body)
        msg['Subject'] = 'TechInterviewerAI: Your Interview Questions and Meeting Link'
        msg['From'] = os.environ["GMAIL_ADDRESS"]
        msg['To'] = recipient

        # Connect to Gmail’s SMTP server
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
            server.login(os.environ["GMAIL_ADDRESS"], os.environ["GMAIL_APP_PASSWORD"])
            server.sendmail(msg['From'], msg['To'], msg.as_string())

        print(f"Email sent successfully to {recipient}")
    except Exception as e:
        print(f"Error sending email: {str(e)}")

## Run Workflow and Send Email

This cell defines the button handler to run the CrewAI workflow, display the generated questions, show a message about the meeting link, and send an email with the meeting link.

In [ ]:
# Define Crew
interview_crew = Crew(
    agents=[
        input_processor_agent,
        search_query_generator,
        research_agent,
        web_scraper_agent,
        question_generator_agent,
    ],
    tasks=[
        input_processor_task,
        search_query_generator_task,
        research_task,
        web_scraper_task,
        question_generator_task,
    ],
    process=Process.sequential
)

# Define button handler
def on_submit_button_clicked(b):
    with output:
        output.clear_output()
        print("Starting agent workflow...")

        # Validate inputs
        if not job_position_input.value or not requirements_input.value or not email_input.value:
            print("Error: Job Position, Requirements, and Email are required.")
            return

        # Run CrewAI workflow
        try:
            result = interview_crew.kickoff(inputs={
                "job_position": job_position_input.value,
                "requirements": requirements_input.value,
                "company_name": company_name_input.value,
                "score_th": score_th_input.value
            })

            # Load and display questions
            final_script_path = os.path.join(output_dir, "step_5_interview_script.json")
            if os.path.exists(final_script_path):
                with open(final_script_path, 'r') as f:
                    interview_script = json.load(f)

                print("\nGenerated Interview Questions:")
                for idx, q in enumerate(interview_script.get("questions", []), 1):
                    print(f"{idx}. {q['question']} (Type: {q['type']}, Difficulty: {q['difficulty']})")

                # Display meeting link message
                print(f"\nA meeting link will be sent to {email_input.value} shortly after generation.")

                # Send email with meeting link
                send_email(email_input.value, interview_script)
            else:
                print("Error: Final interview script not found.")
        except Exception as e:
            print(f"Error running workflow: {str(e)}")

# Attach handler to button
submit_button.on_click(on_submit_button_clicked)